# XFeat: 학습한 float 비트 선택 → 8-bin LUT → ABS / Thermometer

32비트 float 원소에서 **학습으로 고른 8개 위치**를 shift/AND로 추출합니다.
각 차원의 256개 패턴을 **학습한 LUT로 0~7 bin**에 대응시킵니다.

- 정수 방법: bin을 3비트로 저장하고, 정수 ABS를 합산합니다. 촘촘한 저장은 **24 B/descriptor**이며 매칭 시 64 B 작업 배열로 풉니다.
- Thermometer 방법: 같은 bin을 7-bit thermometer로 바꾸고 **7개의 uint64**로 저장합니다. XOR/popcount 합은 같은 ABS 거리입니다. **56 B/descriptor**.
- 선택한 8비트는 **중간 패턴**입니다. 최종 bin은 3비트입니다.
- 전체 64차원에 공통 비트 위치를 쓰고 **차원별 LUT 64×256**을 씁니다. A/B는 같은 규칙을 공유합니다.
- 학습은 greedy search + LUT coordinate descent입니다. XFeat 재학습이나 neural optimizer 학습은 없습니다.
- LUT 초기화/정규화에는 학습 데이터의 분위수 bin을 teacher로 사용합니다. 완전히 분위수와 독립적인 실험은 아닙니다.
- 비트 위치와 매핑은 데이터에 의존합니다. 선행연구 대비 신규성을 확정한 결과가 아닙니다.

## 데이터와 분리
기존 Drive의 `Colab Notebooks/CSE527_datasets/val2017.zip`을 재사용합니다.
seed 20261011로 COCO 원본 이미지 24장을 고릅니다. **16 train / 3 validation / 5 test**.
이미지당 알려진 원근변환 + 밝기변환 3개를 만듭니다.
positive는 양방향 기하학적 최근접 대응이며 양방향 오차 <3 px입니다.
negative는 투영 좌표에서 >10 px 떨어진 후보 중 cosine이 가장 높은 어려운 비대응점입니다.
모든 비트 선택/LUT 학습은 train 이미지에서만 진행합니다.
HPatches는 선택적 외부 평가로만 사용하며 학습에 들어가지 않습니다.

## 실행
위에서 아래로 모두 실행합니다. Drive mount는 Colab의 인증 화면에서 허용해야 합니다.
학습 캐시, codec, 로그, accuracy.csv, timing.csv, results.json은 이 실험 폴더의 `run/`에 저장됩니다.
처음 실행 시 XFeat의 고정된 commit을 내려받습니다. CPU/GPU는 추출에 사용되며 matching은 CPU입니다.


## 실제 CPU 파일럿 결과 — 이미 실행 완료

이 표는 노트북 셀 실행 결과를 가정한 것이 아니라, 함께 제공한 `run/results.json`의 실제 로컬 실행 결과입니다.
선택 비트(LSB=0): **[31, 25, 24, 23, 22, 26, 21, 20]**.
학습 triplet 14,341개, LUT 조정에 최대 8,000개 사용.
비트 선택/LUT 학습 시간 **7.39초**.
독립 test 원본 5장 × 변환 3개 = 15쌍. 아래 precision은 모든 쌍의 correct/mutual을 합산했습니다.

| 방법 | Mutual | Correct (<3 px) | Precision |
|---|---:|---:|---:|
| official_float_cosine | 8567 | 6159 | 71.8921% |
| fixed_quantile_abs | 8240 | 5698 | 69.1505% |
| learned_bits_abs | 8168 | 5587 | 68.4011% |
| learned_bits_thermometer | 8168 | 5587 | 68.4011% |

두 표현의 정확도와 MNN 인덱스는 같아야 합니다. 자동 assert로 검사합니다.
속도는 인코딩을 포함한 값과 미리 인코딩한 값을 모두 기록합니다.
float 기준은 공식 XFeat `match(..., min_cossim=-1)`입니다.
이 C++ matcher는 기본 구현이며 이전 AVX2 SIMD4 커널을 재사용한 결과가 아닙니다.
작은 synthetic-warp 파일럿 결과이므로 HPatches 성능이나 전체 XFeat 가속으로 일반화하지 마세요.


## 1. 의존성 확인

In [ ]:
import importlib.util, subprocess, sys
packages={'cv2':'opencv-python-headless','tqdm':'tqdm','scipy':'scipy','pandas':'pandas'}
for module, package in packages.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',package],check=True)
if importlib.util.find_spec('torch') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','torch','--index-url','https://download.pytorch.org/whl/cpu'],check=True)


## 2. Drive 및 설정

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/2026-10-11_learned_float_bits_8bin')
DATA=Path('/content/drive/MyDrive/Colab Notebooks/CSE527_datasets')
ROOT.mkdir(parents=True,exist_ok=True)
COCO_ZIP=DATA/'val2017.zip'
HPATCHES_ZIP=DATA/'hpatches-sequences-release.zip'
RUN_HPATCHES=False  # True: 학습 후 HPatches 외부 평가도 실행
COUNT=24
WARPS=3
SEARCH_SAMPLES=2000
LUT_SWEEPS=2
assert COCO_ZIP.exists(), f'기존 COCO ZIP을 찾지 못했습니다: {COCO_ZIP}'
if RUN_HPATCHES: assert HPATCHES_ZIP.exists(), str(HPATCHES_ZIP)
print('실험 폴더:', ROOT)


## 3. 구현 복원 및 동일성 검사

In [ ]:
# 두 소스 파일을 이 노트북 자체에서 복원합니다.
PY_SOURCE='"""Learn fixed float-bit positions and ordered 8-bin LUTs; compare exact L1 implementations.\nTraining is discrete greedy selection + supervised coordinate descent, not neural training.\n"""\nfrom pathlib import Path\nimport argparse, ctypes, hashlib, json, platform, subprocess, sys, time, urllib.request, zipfile\nimport numpy as np\n\nSEED=20261011\nXFEAT_COMMIT=\'e92685f57f8318b18725c5c8c0bd28c7fe188d9a\'\n\ndef patterns(x, positions):\n    u=np.ascontiguousarray(x,dtype=np.float32).view(np.uint32)\n    c=np.zeros(u.shape,np.uint16)\n    for r,p in enumerate(positions): c |= (((u >> int(p)) & 1).astype(np.uint16) << r)\n    return c\n\ndef bins_from_edges(x, edges):\n    return (x[:,:,None] >= edges[None,:,:]).sum(-1).astype(np.uint8)\n\ndef initial_lut(x, positions, targets):\n    c=patterns(x,positions); size=1<<len(positions)\n    lut=np.zeros((64,size),np.uint8); counts=np.zeros((64,size),np.int32)\n    for d in range(64):\n        n=np.bincount(c[:,d],minlength=size)\n        s=np.bincount(c[:,d],weights=targets[:,d],minlength=size)\n        # Unseen patterns deterministically map to the training dimension\'s mean bin.\n        means=np.full(size,float(targets[:,d].mean()))\n        np.divide(s,n,out=means,where=n>0)\n        lut[d]=np.clip(np.rint(means),0,7).astype(np.uint8); counts[d]=n\n    return lut,counts\n\ndef encode_np(x,positions,lut): return lut[np.arange(64)[None,:],patterns(x,positions)]\n\ndef triplet_metrics(a,p,n):\n    dp=np.abs(a.astype(np.int16)-p.astype(np.int16)).sum(1)\n    dn=np.abs(a.astype(np.int16)-n.astype(np.int16)).sum(1)\n    return dp,dn,float((dp<dn).mean()),float(np.maximum(0,8+dp-dn).mean()/64)\n\ndef train_codec(triplets, calibration, out, nbits=8, search_samples=2000, sweeps=2):\n    rng=np.random.default_rng(SEED); order=rng.permutation(len(triplets[0]))\n    if len(order)<100: raise ValueError(\'Need at least 100 geometrically labelled triplets.\')\n    cut=max(1,int(len(order)*.2)); inner=order[:cut]; fit=order[cut:]\n    fit=fit[:search_samples]; inner=inner[:search_samples]\n    a,p,n=[np.ascontiguousarray(x[fit],np.float32) for x in triplets]\n    va,vp,vn=[np.ascontiguousarray(x[inner],np.float32) for x in triplets]\n    # All quantile teachers, bit selection and LUT updates use training scenes only.\n    edges=np.quantile(calibration,np.arange(1,8)/8,axis=0).T.astype(np.float32)\n    x=np.concatenate([a,p,n]); target=bins_from_edges(x,edges)\n    vtarget=bins_from_edges(np.concatenate([va,vp,vn]),edges)\n    selected=[]; history=[]; started=time.perf_counter()\n    for step in range(nbits):\n        candidates=[]\n        for bit in range(32):\n            if bit in selected: continue\n            positions=selected+[bit]; lut,_=initial_lut(x,positions,target)\n            codes=encode_np(np.concatenate([va,vp,vn]),positions,lut)\n            ca,cp,cn=np.split(codes,3)\n            _,_,rank,loss=triplet_metrics(ca,cp,cn)\n            distill=float(np.square(codes.astype(np.float32)-vtarget).mean())\n            score=loss+.03*distill\n            candidates.append((score,bit,rank,loss,distill))\n        score,bit,rank,loss,distill=min(candidates)\n        selected.append(bit)\n        row=dict(step=step+1,bit=int(bit),score=score,inner_rank=rank,margin_loss=loss,distill_mse=distill)\n        history.append(row); print(\'bit selection\',row,flush=True)\n        (out/\'selection_history.json\').write_text(json.dumps(history,indent=2))\n    # Refit LUT on all sampled training triplets, then supervise its ordered labels.\n    take=order[:min(len(order),8000)]\n    a,p,n=[np.ascontiguousarray(v[take],np.float32) for v in triplets]\n    x=np.concatenate([a,p,n]); target=bins_from_edges(x,edges)\n    lut,counts=initial_lut(x,selected,target)\n    ca,cp,cn=[patterns(v,selected) for v in (a,p,n)]\n    ya,yp,yn=[lut[np.arange(64)[None,:],c].astype(np.int16) for c in (ca,cp,cn)]\n    ta,tp,tn=np.split(target,3); dp,dn,_,_=triplet_metrics(ya,yp,yn)\n    refinement=[]; regularizer=.03\n    for sweep in range(sweeps):\n        changes=0\n        for d in range(64):\n            for c in rng.permutation(np.flatnonzero(counts[d])):\n                ids=np.flatnonzero((ca[:,d]==c)|(cp[:,d]==c)|(cn[:,d]==c))\n                old=int(lut[d,c]); aa=ya[ids,d]; pp=yp[ids,d]; nn=yn[ids,d]\n                oldp=np.abs(aa-pp); oldn=np.abs(aa-nn)\n                maska=ca[ids,d]==c; maskp=cp[ids,d]==c; maskn=cn[ids,d]==c\n                best=(float(\'inf\'),old,None,None,None,None,None)\n                for val in range(8):\n                    na=np.where(maska,val,aa); np_=np.where(maskp,val,pp); nneg=np.where(maskn,val,nn)\n                    ndp=dp[ids]-oldp+np.abs(na-np_); ndn=dn[ids]-oldn+np.abs(na-nneg)\n                    # This is the affected part of the global triplet loss + teacher regularization.\n                    reg=((na-ta[ids,d])**2+(np_-tp[ids,d])**2+(nneg-tn[ids,d])**2).sum()\n                    score=float(np.maximum(0,8+ndp-ndn).sum()+regularizer*reg/3)\n                    if score<best[0]-1e-9 or (abs(score-best[0])<1e-9 and val==old):\n                        best=(score,val,ndp,ndn,na,np_,nneg)\n                if best[1]!=old:\n                    changes+=1;lut[d,c]=best[1];dp[ids]=best[2];dn[ids]=best[3]\n                    ya[ids,d]=best[4];yp[ids,d]=best[5];yn[ids,d]=best[6]\n        row=dict(sweep=sweep+1,changed_entries=changes,train_rank=float((dp<dn).mean()),\n                 margin_loss=float(np.maximum(0,8+dp-dn).mean()/64))\n        refinement.append(row);print(\'LUT training\',row,flush=True)\n    result=dict(positions=selected,nbits=nbits,levels=8,selection_history=history,refinement=refinement,\n                seconds=time.perf_counter()-started,training_triplets=len(triplets[0]),lut_training_triplets=len(a),\n                method=\'greedy bit selection with internal training-only holdout; supervised integer LUT coordinate descent\',\n                lut_scope=\'per descriptor dimension, shared across all images and keypoints\',\n                teacher=\'training-scene per-dimension quantile bins; regularization only\',\n                unseen_pattern_policy=\'training dimension mean bin; never fitted on evaluation data\')\n    np.savez(out/\'learned_codec.npz\',positions=np.array(selected,np.int32),lut=lut,counts=counts,edges=edges)\n    (out/\'training.json\').write_text(json.dumps(result,indent=2))\n    return selected,lut,edges,result\n\nclass Engine:\n    def __init__(self,root):\n        root=Path(root); lib=root/\'bit_codec.so\'\n        subprocess.run([\'g++\',\'-O3\',\'-march=native\',\'-std=c++17\',\'-shared\',\'-fPIC\',str(root/\'bit_codec.cpp\'),\'-o\',str(lib)],check=True)\n        self.lib=ctypes.CDLL(str(lib.resolve())); self.flags=[\'-O3\',\'-march=native\'];self.root=root\n        P=ctypes.c_void_p; I=ctypes.c_int\n        for name,args in {\'encode\':[P,I,P,I,P,P],\'pack3\':[P,I,P],\'unpack3\':[P,I,P],\n                          \'packthermo\':[P,I,P],\'absdist\':[P,P,I,I,P],\'thermodist\':[P,P,I,I,P],\n                          \'absmnn\':[P,P,I,I,P,P],\'thermomnn\':[P,P,I,I,P,P]}.items():\n            f=getattr(self.lib,name);f.argtypes=args;f.restype=None\n    @staticmethod\n    def ptr(a):return a.ctypes.data_as(ctypes.c_void_p)\n    def encode(self,x,pos,lut):\n        x=np.ascontiguousarray(x,np.float32);pos=np.ascontiguousarray(pos,np.int32);lut=np.ascontiguousarray(lut,np.uint8)\n        out=np.empty(x.shape,np.uint8);self.lib.encode(self.ptr(x),len(x),self.ptr(pos),len(pos),self.ptr(lut),self.ptr(out));return out\n    def pack3(self,x):\n        x=np.ascontiguousarray(x,np.uint8);out=np.empty((len(x),24),np.uint8)\n        self.lib.pack3(self.ptr(x),len(x),self.ptr(out));return out\n    def unpack3(self,x):\n        x=np.ascontiguousarray(x,np.uint8);out=np.empty((len(x),64),np.uint8)\n        self.lib.unpack3(self.ptr(x),len(x),self.ptr(out));return out\n    def thermo(self,x):\n        x=np.ascontiguousarray(x,np.uint8);out=np.empty((len(x),7),np.uint64)\n        self.lib.packthermo(self.ptr(x),len(x),self.ptr(out));return out\n    def distance(self,a,b,thermo=False):\n        out=np.empty((len(a),len(b)),np.uint16)\n        getattr(self.lib,\'thermodist\' if thermo else \'absdist\')(self.ptr(a),self.ptr(b),len(a),len(b),self.ptr(out));return out\n    def mnn(self,a,b,thermo=False):\n        if len(a)==0 or len(b)==0:return np.empty(0,int),np.empty(0,int)\n        row=np.empty(len(a),np.int32);col=np.empty(len(b),np.int32)\n        getattr(self.lib,\'thermomnn\' if thermo else \'absmnn\')(self.ptr(a),self.ptr(b),len(a),len(b),self.ptr(row),self.ptr(col))\n        ids=np.flatnonzero(col[row]==np.arange(len(a)));return ids,row[ids]\n\ndef self_test(engine):\n    rng=np.random.default_rng(SEED)\n    a=rng.integers(0,8,(35,64),dtype=np.uint8);b=rng.integers(0,8,(31,64),dtype=np.uint8)\n    # Force ties to verify deterministic first-index MNN rules.\n    a[1]=a[0];b[1]=b[0]\n    assert np.array_equal(a,engine.unpack3(engine.pack3(a)))\n    expected=np.abs(a[:,None,:].astype(np.int16)-b[None,:,:].astype(np.int16)).sum(-1)\n    assert np.array_equal(engine.distance(a,b),expected)\n    at,bt=engine.thermo(a),engine.thermo(b)\n    assert np.array_equal(engine.distance(at,bt,True),expected)\n    row=expected.argmin(1);col=expected.argmin(0);ids=np.flatnonzero(col[row]==np.arange(len(a)))\n    for aa,bb,t in [(a,b,False),(at,bt,True)]:\n        i,j=engine.mnn(aa,bb,t);assert np.array_equal(i,ids) and np.array_equal(j,row[ids])\n    # Exhaustively test all bin pairs and the sign bit extraction.\n    q=np.repeat(np.arange(8,dtype=np.uint8)[:,None],64,axis=1)\n    assert np.array_equal(engine.distance(engine.thermo(q),engine.thermo(q),True),64*np.abs(np.arange(8)[:,None]-np.arange(8)[None,:]))\n    x=rng.normal(size=(20,64)).astype(np.float32);pos=[31,25,24,23,22,21,20,19]\n    lut=rng.integers(0,8,(64,256),dtype=np.uint8)\n    assert np.array_equal(engine.encode(x,pos,lut),encode_np(x,pos,lut))\n    print(\'PASS: bit extraction, 3-bit roundtrip, all-bin ABS/popcount equality, full matrices, MNN including ties\',flush=True)\n\ndef project(k,H):\n    p=np.column_stack([k,np.ones(len(k))])@H.T\n    return (p[:,:2]/p[:,2:]).astype(np.float32)\n\ndef geometric_pairs(pair):\n    from scipy.spatial.distance import cdist\n    d=cdist(project(pair[\'k1\'],pair[\'H\']),pair[\'k2\']);back=cdist(project(pair[\'k2\'],np.linalg.inv(pair[\'H\'])),pair[\'k1\'])\n    j=d.argmin(1);i=np.arange(len(j));good=(d[i,j]<3)&(back[j,i]<3)&(back.argmin(1)[j]==i)\n    return i[good],j[good],d\n\ndef build_triplets(pairs):\n    result=[[],[],[]];cal=[];rng=np.random.default_rng(SEED)\n    for pair in pairs:\n        i,j,geo=geometric_pairs(pair)\n        if not len(i):continue\n        if len(i)>300:\n            sub=rng.choice(len(i),300,False);i,j=i[sub],j[sub]\n        # Hard negatives: most cosine-similar target at least 10 px from projected anchor.\n        sim=pair[\'d1\'][i]@pair[\'d2\'].T;sim[geo[i]<=10]=-np.inf\n        keep=np.isfinite(sim.max(1));i,j,sim=i[keep],j[keep],sim[keep]\n        if not len(i):continue\n        neg=sim.argmax(1)\n        for dest,x in zip(result,[pair[\'d1\'][i],pair[\'d2\'][j],pair[\'d2\'][neg]]):dest.append(x)\n        cal.extend([pair[\'d1\'],pair[\'d2\']])\n    if not result[0]:raise RuntimeError(\'No valid geometric triplets; inspect images and extraction.\')\n    return tuple(np.concatenate(x) for x in result),np.concatenate(cal)\n\ndef load_xfeat(root,top_k=1000):\n    import torch\n    root=Path(root);repo=root/\'xfeat_repo\'\n    if not repo.exists():\n        subprocess.run([\'git\',\'clone\',\'https://github.com/verlab/accelerated_features.git\',str(repo)],check=True)\n        subprocess.run([\'git\',\'-C\',str(repo),\'checkout\',XFEAT_COMMIT],check=True)\n    commit=subprocess.check_output([\'git\',\'-C\',str(repo),\'rev-parse\',\'HEAD\'],text=True).strip()\n    if commit!=XFEAT_COMMIT:raise RuntimeError(\'XFeat checkout differs from pinned experiment commit.\')\n    sys.path.insert(0,str(repo.resolve()))\n    from modules.xfeat import XFeat\n    torch.set_num_threads(2);torch.manual_seed(SEED)\n    model=XFeat(top_k=top_k)\n    for p in model.parameters():p.requires_grad_(False)\n    return model,commit\n\ndef extract(model,image):\n    import torch,cv2\n    t=torch.from_numpy(cv2.cvtColor(image,cv2.COLOR_BGR2RGB).copy()).permute(2,0,1)[None].float().to(model.dev)/255.\n    with torch.inference_mode():f=model.detectAndCompute(t,top_k=1000)[0]\n    d=f[\'descriptors\'].cpu().numpy().astype(np.float32)\n    d/=np.maximum(np.linalg.norm(d,axis=1,keepdims=True),1e-12)\n    return f[\'keypoints\'].cpu().numpy().astype(np.float32),np.ascontiguousarray(d)\n\ndef make_pairs(images,model,warps,seed):\n    import cv2\n    rng=np.random.default_rng(seed);pairs=[]\n    for name,img in images:\n        h,w=img.shape[:2];k1,d1=extract(model,img)\n        for r in range(warps):\n            aff=cv2.getRotationMatrix2D((w/2,h/2),float(rng.uniform(-20,20)),float(rng.uniform(.88,1.08)))\n            aff[:,2]+=rng.uniform(-.06,.06,2)*[w,h]\n            H=np.vstack([aff,[rng.uniform(-7e-5,7e-5),rng.uniform(-7e-5,7e-5),1]]).astype(np.float32)\n            im=cv2.warpPerspective(img,H,(w,h),borderMode=cv2.BORDER_CONSTANT)\n            im=np.clip(im.astype(np.float32)*rng.uniform(.82,1.12)+rng.uniform(-12,18),0,255).astype(np.uint8)\n            k2,d2=extract(model,im)\n            if min(len(d1),len(d2))<2:continue\n            pairs.append(dict(name=f\'{name}/warp{r}\',d1=d1,d2=d2,k1=k1,k2=k2,H=H))\n        print(\'extract\',name,len(d1),flush=True)\n    return pairs\n\ndef resize_decode(raw,max_side=640):\n    import cv2\n    im=cv2.imdecode(np.frombuffer(raw,np.uint8),cv2.IMREAD_COLOR)\n    if im is None:raise ValueError(\'Unreadable image\')\n    h,w=im.shape[:2];s=min(1,max_side/max(h,w));return cv2.resize(im,(max(64,int(w*s)//32*32),max(64,int(h*s)//32*32)))\n\ndef read_coco(zip_path,count=24):\n    with zipfile.ZipFile(zip_path) as z:\n        names=sorted(n for n in z.namelist() if n.lower().endswith(\'.jpg\'))\n        if len(names)<count:raise ValueError(\'Not enough COCO images\')\n        chosen=np.random.default_rng(SEED).choice(len(names),count,False)\n        return [(Path(names[i]).name,resize_decode(z.read(names[i]))) for i in chosen]\n\ndef read_local_images(folder,count=24):\n    files=sorted(p for p in Path(folder).glob(\'*\') if p.suffix.lower() in [\'.jpg\',\'.jpeg\',\'.png\'])\n    if len(files)<count:raise ValueError(f\'Need {count} distinct images; found {len(files)}\')\n    return [(p.name,resize_decode(p.read_bytes())) for p in files[:count]]\n\ndef save_pair_cache(path,pairs):\n    payload={}\n    for i,p in enumerate(pairs):\n        for k,v in p.items():payload[f\'{i}_{k}\']=np.array(v)\n    payload[\'count\']=np.array(len(pairs));np.savez_compressed(path,**payload)\n\ndef read_pair_cache(path):\n    with np.load(path,allow_pickle=False) as z:\n        return [{k:(str(z[f\'{i}_{k}\']) if k==\'name\' else z[f\'{i}_{k}\']) for k in [\'name\',\'d1\',\'d2\',\'k1\',\'k2\',\'H\']} for i in range(int(z[\'count\']))]\n\ndef benchmark(fn,repeats=9):\n    for _ in range(3):fn()\n    vals=[]\n    for _ in range(repeats):\n        s=time.perf_counter();fn();vals.append((time.perf_counter()-s)*1000)\n    return float(np.median(vals))\n\ndef evaluate(pairs,model,engine,pos,lut,edges,out,label=\'heldout\'):\n    import torch\n    records=[]\n    for p in pairs:\n        a,b=[engine.encode(p[k],pos,lut) for k in [\'d1\',\'d2\']]\n        aa,bb=engine.pack3(a),engine.pack3(b);at,bt=engine.thermo(a),engine.thermo(b)\n        mi,mj=engine.mnn(a,b);ti,tj=engine.mnn(at,bt,True)\n        assert np.array_equal(mi,ti) and np.array_equal(mj,tj)\n        assert np.array_equal(engine.distance(a[:32],b[:32]),engine.distance(at[:32],bt[:32],True))\n        fa,fb=torch.from_numpy(p[\'d1\']),torch.from_numpy(p[\'d2\'])\n        def official():\n            with torch.inference_mode():return model.match(fa,fb,min_cossim=-1)\n        fi,fj=official();fi,fj=fi.numpy(),fj.numpy()\n        qa,qb=[bins_from_edges(p[k],edges) for k in [\'d1\',\'d2\']];qi,qj=engine.mnn(qa,qb)\n        for method,i,j in [(\'official_float_cosine\',fi,fj),(\'fixed_quantile_abs\',qi,qj),(\'learned_bits_abs\',mi,mj),(\'learned_bits_thermometer\',ti,tj)]:\n            err=np.linalg.norm(project(p[\'k1\'][i],p[\'H\'])-p[\'k2\'][j],axis=1)\n            gt1,gt2,_=geometric_pairs(p);gt=set(zip(gt1.tolist(),gt2.tolist()));hits=len(gt&set(zip(i.tolist(),j.tolist())))\n            records.append(dict(dataset=label,pair=p[\'name\'],method=method,n=len(a),m=len(b),mutual=len(i),correct=int((err<3).sum()),\n                precision=float((err<3).mean()*100) if len(err) else 0.,median_px=float(np.median(err)) if len(err) else None,\n                geometric_pair_recall=float(hits/len(gt)) if gt else None))\n        def encode_abs():\n            a,b=[engine.encode(p[k],pos,lut) for k in [\'d1\',\'d2\']];return engine.mnn(a,b)\n        def encode_packed3():\n            a,b=[engine.pack3(engine.encode(p[k],pos,lut)) for k in [\'d1\',\'d2\']]\n            return engine.mnn(engine.unpack3(a),engine.unpack3(b))\n        def encode_thermo():\n            a,b=[engine.thermo(engine.encode(p[k],pos,lut)) for k in [\'d1\',\'d2\']];return engine.mnn(a,b,True)\n        timings=dict(pair=p[\'name\'],n=len(a),m=len(b),official_float_mnn_ms=benchmark(official),\n            bin_encode_both_ms=benchmark(lambda:(engine.encode(p[\'d1\'],pos,lut),engine.encode(p[\'d2\'],pos,lut))),\n            abs_preencoded_mnn_ms=benchmark(lambda:engine.mnn(a,b)),\n            abs_packed3_preencoded_mnn_including_unpack_ms=benchmark(lambda:engine.mnn(engine.unpack3(aa),engine.unpack3(bb))),\n            thermometer_preencoded_mnn_ms=benchmark(lambda:engine.mnn(at,bt,True)),\n            abs_encode_mnn_ms=benchmark(encode_abs),abs_encode_pack_unpack_mnn_ms=benchmark(encode_packed3),\n            thermometer_encode_pack_mnn_ms=benchmark(encode_thermo))\n        for r in records[-4:]:r[\'timings\']=timings\n        print(\'eval\',label,p[\'name\'],[(r[\'method\'],round(r[\'precision\'],2),r[\'mutual\']) for r in records[-4:]],flush=True)\n    (out/f\'{label}_results.json\').write_text(json.dumps(records,indent=2));return records\n\ndef hpatches_pairs(zip_path,model,max_sequences=8):\n    import cv2\n    pairs=[]\n    with zipfile.ZipFile(zip_path) as z:\n        names=set(z.namelist());refs=sorted(n for n in names if n.endswith(\'/1.ppm\'))[:max_sequences]\n        for ref in refs:\n            directory=ref.rsplit(\'/\',1)[0];im1=cv2.imdecode(np.frombuffer(z.read(ref),np.uint8),cv2.IMREAD_COLOR)\n            k1,d1=extract(model,im1)\n            for index in range(2,7):\n                imname=f\'{directory}/{index}.ppm\';hname=f\'{directory}/H_1_{index}\'\n                if imname not in names or hname not in names:continue\n                im2=cv2.imdecode(np.frombuffer(z.read(imname),np.uint8),cv2.IMREAD_COLOR);k2,d2=extract(model,im2)\n                H=np.loadtxt(z.open(hname)).astype(np.float32)\n                if min(len(d1),len(d2))>=2:pairs.append(dict(name=f\'{directory}/1-{index}\',d1=d1,d2=d2,k1=k1,k2=k2,H=H))\n    if not pairs:raise ValueError(\'HPatches archive missing expected sequence images/homographies\')\n    return pairs\n\ndef run(root,coco_zip=None,image_dir=None,count=24,warps=3,search_samples=2000,sweeps=2,hpatches=None):\n    root=Path(root);root.mkdir(parents=True,exist_ok=True);out=root/\'run\';out.mkdir(exist_ok=True)\n    engine=Engine(root);self_test(engine);model,commit=load_xfeat(root)\n    images=sorted(read_coco(coco_zip,count) if coco_zip else read_local_images(image_dir,count),key=lambda v:v[0])\n    order=np.random.default_rng(SEED).permutation(count);nt=max(2,int(count*.67));nv=max(1,int(count*.16))\n    groups=dict(train=order[:nt],validation=order[nt:nt+nv],test=order[nt+nv:])\n    manifest=dict(seed=SEED,source=\'COCO val2017\',groups={k:[images[i][0] for i in ids] for k,ids in groups.items()},\n                  xfeat_commit=commit,top_k=1000,max_side=640,warps_per_image=warps,\n                  note=\'image-disjoint synthetic homographies; COCO val2017 is our training source, not a COCO official benchmark\')\n    manifest[\'cache_fingerprint\']=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()+b\'\'.join(hashlib.sha256(im.tobytes()).digest() for _,im in images)).hexdigest()\n    manifest_path=out/\'data_manifest.json\'\n    if manifest_path.exists() and json.loads(manifest_path.read_text()).get(\'cache_fingerprint\')!=manifest[\'cache_fingerprint\']:\n        raise RuntimeError(\'Existing pair cache belongs to a different dataset/config. Use a new experiment root or remove only its run/*_pairs.npz and data_manifest.json.\')\n    manifest_path.write_text(json.dumps(manifest,indent=2))\n    pairs={}\n    extraction=time.perf_counter()\n    for offset,(name,ids) in enumerate(groups.items()):\n        cache=out/f\'{name}_pairs.npz\'\n        if cache.exists():pairs[name]=read_pair_cache(cache)\n        else:\n            pairs[name]=make_pairs([images[i] for i in ids],model,warps,SEED+offset);save_pair_cache(cache,pairs[name])\n    extraction=time.perf_counter()-extraction\n    triplets,cal=build_triplets(pairs[\'train\'])\n    np.savez_compressed(out/\'training_triplets.npz\',anchor=triplets[0],positive=triplets[1],negative=triplets[2])\n    pos,lut,edges,training=train_codec(triplets,cal,out,search_samples=search_samples,sweeps=sweeps)\n    assert np.array_equal(engine.encode(cal[:100],pos,lut),encode_np(cal[:100],pos,lut))\n    records=[]\n    for name in [\'validation\',\'test\']:records+=evaluate(pairs[name],model,engine,pos,lut,edges,out,name)\n    hp_status=\'not run\'\n    if hpatches:\n        records+=evaluate(hpatches_pairs(hpatches,model),model,engine,pos,lut,edges,out,\'hpatches\');hp_status=\'run, never used for training\'\n    report=dict(training=training,manifest=manifest,extraction_seconds=extraction,hpatches=hp_status,\n        environment=dict(platform=platform.platform(),python=sys.version,numpy=np.__version__,cpp_flags=engine.flags,cpp_threads=1,torch_threads=2),\n        storage_bytes=dict(float32=256,selected_bit_intermediate=64,bin_uint8_working=64,bin_packed3=24,thermometer_packed=56,\n                           shared_lut=lut.nbytes,selected_positions=np.array(pos,np.int32).nbytes),\n        accuracy_equivalence=\'ABS and thermometer must match exactly, including deterministic tie-breaking\',\n        timing_scope=\'CPU encode/pack/matching; XFeat extraction excluded and reported separately; no claimed SIMD4 optimization\',\n        records=records)\n    (out/\'results.json\').write_text(json.dumps(report,indent=2))\n    import csv\n    with (out/\'accuracy.csv\').open(\'w\',newline=\'\') as f:\n        rows=[{k:v for k,v in r.items() if k!=\'timings\'} for r in records];w=csv.DictWriter(f,rows[0].keys());w.writeheader();w.writerows(rows)\n    with (out/\'timing.csv\').open(\'w\',newline=\'\') as f:\n        rows=[dict(dataset=r[\'dataset\'],**r[\'timings\']) for r in records if r[\'method\']==\'learned_bits_abs\']\n        w=csv.DictWriter(f,rows[0].keys());w.writeheader();w.writerows(rows)\n    print(\'Saved:\',out,flush=True);return report\n\nif __name__==\'__main__\':\n    ap=argparse.ArgumentParser();ap.add_argument(\'--root\',default=str(Path(__file__).parent));ap.add_argument(\'--coco-zip\');ap.add_argument(\'--image-dir\')\n    ap.add_argument(\'--count\',type=int,default=24);ap.add_argument(\'--warps\',type=int,default=3);ap.add_argument(\'--search-samples\',type=int,default=2000)\n    ap.add_argument(\'--sweeps\',type=int,default=2);ap.add_argument(\'--hpatches\');ap.add_argument(\'--self-test\',action=\'store_true\');args=ap.parse_args()\n    if args.self_test:self_test(Engine(args.root))\n    else:run(args.root,args.coco_zip,args.image_dir,args.count,args.warps,args.search_samples,args.sweeps,args.hpatches)\n'
CPP_SOURCE='#include <cstdint>\n#include <cstring>\n#include <vector>\n#include <cstdlib>\n#include <algorithm>\nextern "C" {\nvoid encode(const float* x,int n,const int* pos,int k,const uint8_t* lut,uint8_t* out){\n for(int i=0;i<n;i++) for(int d=0;d<64;d++){\n  uint32_t u; std::memcpy(&u,x+i*64+d,4); unsigned c=0;\n  for(int r=0;r<k;r++) c|=((u>>pos[r])&1u)<<r;\n  out[i*64+d]=lut[d*(1<<k)+c];\n }\n}\nvoid pack3(const uint8_t* x,int n,uint8_t* out){\n std::memset(out,0,n*24);\n for(int i=0;i<n;i++) for(int d=0;d<64;d++){\n  unsigned b=d*3, v=x[i*64+d]; out[i*24+b/8]|=v<<(b%8);\n  if(b%8>5) out[i*24+b/8+1]|=v>>(8-b%8);\n }\n}\nvoid unpack3(const uint8_t* x,int n,uint8_t* out){\n for(int i=0;i<n;i++) for(int d=0;d<64;d++){\n  unsigned b=d*3,v=x[i*24+b/8]>>(b%8);\n  if(b%8>5) v|=unsigned(x[i*24+b/8+1])<<(8-b%8);\n  out[i*64+d]=v&7;\n }\n}\nvoid packthermo(const uint8_t* x,int n,uint64_t* out){\n for(int i=0;i<n;i++) for(int r=0;r<7;r++){\n  uint64_t mask=0;for(int d=0;d<64;d++) if(x[i*64+d]>r) mask|=uint64_t(1)<<d;\n  out[i*7+r]=mask;\n }\n}\nvoid absdist(const uint8_t* a,const uint8_t* b,int n,int m,uint16_t* out){\n for(int i=0;i<n;i++)for(int j=0;j<m;j++){\n  unsigned s=0;for(int d=0;d<64;d++)s+=std::abs(int(a[i*64+d])-int(b[j*64+d]));out[i*m+j]=s;\n }\n}\nvoid thermodist(const uint64_t* a,const uint64_t* b,int n,int m,uint16_t* out){\n for(int i=0;i<n;i++)for(int j=0;j<m;j++){\n  unsigned s=0;for(int r=0;r<7;r++)s+=__builtin_popcountll(a[i*7+r]^b[j*7+r]);out[i*m+j]=s;\n }\n}\nvoid absmnn(const uint8_t* a,const uint8_t* b,int n,int m,int* row,int* col){\n std::vector<unsigned> bestrow(n,10000),bestcol(m,10000);\n std::fill(row,row+n,-1);std::fill(col,col+m,-1);\n for(int i=0;i<n;i++)for(int j=0;j<m;j++){\n  unsigned s=0;for(int d=0;d<64;d++)s+=std::abs(int(a[i*64+d])-int(b[j*64+d]));\n  if(s<bestrow[i]){bestrow[i]=s;row[i]=j;}if(s<bestcol[j]){bestcol[j]=s;col[j]=i;}\n }\n}\nvoid thermomnn(const uint64_t* a,const uint64_t* b,int n,int m,int* row,int* col){\n std::vector<unsigned> bestrow(n,10000),bestcol(m,10000);\n std::fill(row,row+n,-1);std::fill(col,col+m,-1);\n for(int i=0;i<n;i++)for(int j=0;j<m;j++){\n  unsigned s=0;for(int r=0;r<7;r++)s+=__builtin_popcountll(a[i*7+r]^b[j*7+r]);\n  if(s<bestrow[i]){bestrow[i]=s;row[i]=j;}if(s<bestcol[j]){bestcol[j]=s;col[j]=i;}\n }\n}\n}\n'
(ROOT/'experiment.py').write_text(PY_SOURCE)
(ROOT/'bit_codec.cpp').write_text(CPP_SOURCE)
import importlib.util
spec=importlib.util.spec_from_file_location('learned_float_bits_experiment',ROOT/'experiment.py')
experiment=importlib.util.module_from_spec(spec)
spec.loader.exec_module(experiment)
experiment.self_test(experiment.Engine(ROOT))


## 4. 실제 추출 → 학습 → 검증 → 테스트

In [ ]:
import json, io, contextlib
log=io.StringIO()
class Tee:
    def __init__(self,*streams):self.streams=streams
    def write(self,s):
        for stream in self.streams:stream.write(s)
        return len(s)
    def flush(self):
        for stream in self.streams:stream.flush()
import sys
try:
    with contextlib.redirect_stdout(Tee(sys.stdout,log)):
        report=experiment.run(ROOT,coco_zip=COCO_ZIP,count=COUNT,warps=WARPS,
            search_samples=SEARCH_SAMPLES,sweeps=LUT_SWEEPS,
            hpatches=HPATCHES_ZIP if RUN_HPATCHES else None)
finally:
    (ROOT/'training_run.log').write_text(log.getvalue())


## 5. 결과와 시간 비교

In [ ]:
import pandas as pd, numpy as np
from IPython.display import display
records=pd.read_csv(ROOT/'run/accuracy.csv')
summary=records.groupby(['dataset','method'])[['mutual','correct']].sum().reset_index()
summary['precision_percent']=100*summary['correct']/summary['mutual']
display(summary)
timing=pd.read_csv(ROOT/'run/timing.csv')
display(timing.groupby('dataset').median(numeric_only=True))
print('Selected bits:',report['training']['positions'])
print('Codec:', ROOT/'run/learned_codec.npz')
print('Training:', ROOT/'run/training.json')
print('Image split:', ROOT/'run/data_manifest.json')
print('Results:', ROOT/'run/results.json')
print('Storage:', report['storage_bytes'])


## 6. 저장된 모델로 인코딩

In [ ]:
# 재학습 없이 저장된 codec을 읽어 인코딩하기
codec=np.load(ROOT/'run/learned_codec.npz',allow_pickle=False)
engine=experiment.Engine(ROOT)
pair=experiment.read_pair_cache(ROOT/'run/test_pairs.npz')[0]
a=engine.encode(pair['d1'],codec['positions'],codec['lut'])
b=engine.encode(pair['d2'],codec['positions'],codec['lut'])
packed3_a=engine.pack3(a)
thermo_a=engine.thermo(a);thermo_b=engine.thermo(b)
abs_i,abs_j=engine.mnn(a,b)
thermo_i,thermo_j=engine.mnn(thermo_a,thermo_b,True)
assert np.array_equal(abs_i,thermo_i) and np.array_equal(abs_j,thermo_j)
print('원소별 bin:',a[0])
print('3-bit packed shape:',packed3_a.shape,'Thermometer shape:',thermo_a.shape)
